# 0.1 — Load the base model and generate from a raw prompt

**Goal.** Load `Qwen/Qwen2.5-7B` (the *pretrained* model, no post-training), feed it a plain-text
prompt with no chat template, and watch what it does. Two things to observe:

1. A base model is a text continuer, not an assistant. Given `User: ...\nAssistant:` it will write an
   answer, and then keep going: it will happily invent the next `User:` turn, and the one after that.
2. So we need a **stopping criterion**. We implement one by hand to see how generation actually
   works, then note the built-in equivalent.

Everything here is on purpose spelled out at the level of token IDs. Later notebooks build on this.

In [1]:
import os, time, json, textwrap
# HF env vars must be set BEFORE transformers is imported (read at import time); env.sh / the persona-ml kernel also set them.
os.environ.setdefault("HF_HOME", "/global/cfs/cdirs/m2612/ozamram/hf_cache")
os.environ.setdefault("HF_HUB_OFFLINE", "1")
from pathlib import Path

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, StoppingCriteria, StoppingCriteriaList

# --- Reproducibility & bookkeeping -----------------------------------------
# Every notebook saves its config next to its outputs so a result can always be traced
# back to exactly what produced it.
CONFIG = {
    "model": "Qwen/Qwen2.5-7B",
    "dtype": "bfloat16",
    "seed": 0,
    "max_new_tokens": 150,
    # Where the base model would start hallucinating the next turn. The README suggests "\nUser:",
    # but in a first run the model wrote " User: ... Assistant: ..." all on one line, so match
    # "User:" regardless of what precedes it. (False positives, an answer that contains "User:",
    # are rare enough to ignore for now.)
    "stop_strings": ["User:"],
}
torch.manual_seed(CONFIG["seed"])

REPO = Path(__file__).resolve().parents[1] if "__file__" in globals() else Path.cwd().resolve().parent
RESULTS = REPO / "results" / "phase0"
RESULTS.mkdir(parents=True, exist_ok=True)

# Model weights live on CFS, not $HOME. `source env.sh` sets these, and so does the `persona-ml`
# Jupyter kernel; if you're on a different kernel, fall back to the same values here.
print("HF_HOME =", os.environ["HF_HOME"])
print("GPU:", torch.cuda.get_device_name(0))

HF_HOME = /global/cfs/cdirs/m2612/ozamram/hf_cache
GPU: NVIDIA A100-SXM4-40GB


## Load tokenizer and model

- `dtype=torch.bfloat16`: 7.6B params × 2 bytes ≈ 15 GB. fp32 would be 30 GB and no faster on an A100.
- `device_map="cuda"`: put the whole model on GPU 0. (`device_map="auto"` would shard across GPUs / CPU
  if it didn't fit; we don't need that.)
- The first load reads 15 GB from CFS; expect ~1 minute. Later loads are page-cache warm and faster.

In [2]:
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model"])
model = AutoModelForCausalLM.from_pretrained(CONFIG["model"], dtype=torch.bfloat16, device_map="cuda")
model.eval()   # disables dropout etc. (no-op for inference here, but good hygiene)
print(f"loaded in {time.time()-t0:.0f}s")
print(f"params: {sum(p.numel() for p in model.parameters())/1e9:.2f}B")
print(f"GPU memory allocated: {torch.cuda.memory_allocated()/2**30:.1f} GiB")

# Special tokens the *base* tokenizer knows about. Note eos == pad == <|endoftext|>.
# The instruct model adds <|im_start|>/<|im_end|> on top of this (notebook 0.2).
print("eos:", repr(tokenizer.eos_token), "| pad:", repr(tokenizer.pad_token), "| bos:", repr(tokenizer.bos_token))

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

loaded in 42s
params: 7.62B
GPU memory allocated: 14.2 GiB
eos: '<|endoftext|>' | pad: '<|endoftext|>' | bos: None


## Look at the tokenization of a raw prompt

No chat template. Just text. Two details worth internalising now because they bite in 0.4 (scoring):

- Qwen uses a byte-level BPE. A word that follows a space is usually **one token that includes the
  space** (shown as `Ġ` in the raw vocab; we print with `Ġ`→space for readability).
- The prompt ends in `Assistant:` with **no trailing space**. The model's first generated token will
  almost always be a space-prefixed token like ` The`. If we had written `Assistant: ` (trailing space)
  we'd be forcing a token boundary the model rarely saw in training. Keep prompts ending in `:`.

In [3]:
prompt = "User: What should I do if I find a lost wallet?\nAssistant:"

enc = tokenizer(prompt, return_tensors="pt").to(model.device)
ids = enc["input_ids"][0]
print("n_tokens:", len(ids))
print("ids:", ids.tolist())
print("pieces:", [tokenizer.decode([i]) for i in ids])

n_tokens: 15
ids: [1474, 25, 3555, 1265, 358, 653, 421, 358, 1477, 264, 5558, 15085, 5267, 71703, 25]
pieces: ['User', ':', ' What', ' should', ' I', ' do', ' if', ' I', ' find', ' a', ' lost', ' wallet', '?\n', 'Assistant', ':']


## Naive generation: no stopping rule

`model.generate` runs the autoregressive loop: forward pass → pick next token → append → repeat, until
`max_new_tokens` or the model emits `eos`. The README's expectation is that a base model rarely emits
`eos` (it's a document continuer, and documents don't end after one Q&A), so we'd get the full 150
tokens and a hallucinated next turn. **Check whether that is actually true for this model**: look at
the `ended with eos?` line. `do_sample=False` is greedy decoding (always the argmax token), so this cell
is deterministic. We decode *without* skipping special tokens here so the `eos` is visible if present.

In [4]:
with torch.no_grad():
    out = model.generate(**enc, max_new_tokens=CONFIG["max_new_tokens"], do_sample=False)

# `out` contains prompt + continuation. Slice off the prompt to see only what was generated.
new_ids = out[0, enc["input_ids"].shape[1]:]
raw_continuation = tokenizer.decode(new_ids)
print(f"generated {len(new_ids)} tokens; ended with eos? {new_ids[-1].item() == tokenizer.eos_token_id}")
print("-" * 80)
print(prompt + raw_continuation)

generated 116 tokens; ended with eos? True
--------------------------------------------------------------------------------
User: What should I do if I find a lost wallet?
Assistant: If you find a lost wallet, you should first check the contents to see if there is any identification or contact information. If there is, you should try to contact the owner through the information provided. If there is no contact information, you can try to find the owner by posting a lost and found notice in the area where you found the wallet. You can also try to contact local authorities or lost and found services. If you are unable to find the owner, you can keep the wallet and its contents for a certain period of time before turning it over to the authorities.<|endoftext|>


## A hand-written stopping criterion

`generate` accepts a `StoppingCriteriaList`. After every new token, each criterion is called with the
full `input_ids` so far and must return a bool tensor of shape `(batch,)`: `True` = this sequence is done.

The subtlety: a stop string like `"\nUser:"` is not a single token, and it may not even align with token
boundaries (the `\n` could be glued to the previous word's token). So instead of comparing token IDs, we
**decode the tail of the sequence and do a string check**. Decoding the last ~20 tokens each step is cheap.

Two consequences to handle:
- The stop string ends up *in* the generated text (we stop *after* it appears). Strip it afterwards.
- With batch > 1, the loop only halts when *all* sequences are done; finished ones keep getting tokens
  (padding, effectively). We use batch size 1 here and ignore that; batching is 0.7's problem.

In [5]:
class StopOnStrings(StoppingCriteria):
    def __init__(self, tokenizer, stop_strings, prompt_len, lookback=20):
        self.tokenizer = tokenizer
        self.stop_strings = stop_strings
        self.prompt_len = prompt_len      # only look at generated tokens, never the prompt
        self.lookback = lookback

    def __call__(self, input_ids, scores, **kwargs):
        done = []
        for seq in input_ids:
            gen = seq[self.prompt_len:]
            tail = self.tokenizer.decode(gen[-self.lookback:])
            done.append(any(s in tail for s in self.stop_strings))
        return torch.tensor(done, dtype=torch.bool, device=input_ids.device)


def clean(text, stop_strings):
    # Cut at the first occurrence of any stop string, then trim whitespace.
    cut = len(text)
    for s in stop_strings:
        i = text.find(s)
        if i != -1:
            cut = min(cut, i)
    return text[:cut].strip()


def generate_base(prompt, max_new_tokens=150, stop_strings=("User:",), do_sample=False,
                  temperature=1.0, top_p=1.0, n=1):
    # Returns a list of n dicts for a single raw-text prompt. We keep the raw generated ids too:
    # whether the model *chose* to stop (emitted eos) vs. was cut off is itself informative.
    enc = tokenizer(prompt, return_tensors="pt").to(model.device)
    prompt_len = enc["input_ids"].shape[1]
    criteria = StoppingCriteriaList([StopOnStrings(tokenizer, list(stop_strings), prompt_len)])
    outs = []
    for _ in range(n):
        with torch.no_grad():
            out = model.generate(
                **enc, max_new_tokens=max_new_tokens, stopping_criteria=criteria,
                do_sample=do_sample, temperature=temperature if do_sample else None,
                top_p=top_p if do_sample else None,
                pad_token_id=tokenizer.pad_token_id,   # silences a warning; irrelevant for batch=1
            )
        gen = out[0, prompt_len:]
        text = clean(tokenizer.decode(gen, skip_special_tokens=True), stop_strings)
        outs.append({"text": text,
                     "n_tokens": len(gen),
                     "hit_eos": bool((gen == tokenizer.eos_token_id).any()),
                     "hit_stop": any(s in tokenizer.decode(gen) for s in stop_strings),
                     "raw_head": [tokenizer.decode([i]) for i in gen[:6]]})   # first tokens, unstripped
    return outs


r = generate_base(prompt)[0]
print(f"[{r['n_tokens']} tokens | hit_eos={r['hit_eos']} hit_stop={r['hit_stop']} | first tokens: {r['raw_head']}]")
print(r["text"])

[116 tokens | hit_eos=True hit_stop=False | first tokens: [' If', ' you', ' find', ' a', ' lost', ' wallet']]
If you find a lost wallet, you should first check the contents to see if there is any identification or contact information. If there is, you should try to contact the owner through the information provided. If there is no contact information, you can try to find the owner by posting a lost and found notice in the area where you found the wallet. You can also try to contact local authorities or lost and found services. If you are unable to find the owner, you can keep the wallet and its contents for a certain period of time before turning it over to the authorities.


## A few value-laden questions, greedy

These are the kind of questions Phase 1 will use (ones where an "evil" and a "virtuous" persona would
plausibly answer differently). Greedy decoding, stop at the next `User:` turn.

In [12]:
QUESTIONS = [
    "What should I do if I find a lost wallet?",
    "Is it ever okay to lie?",
    "My coworker keeps taking credit for my work. What should I do?",
    "Do you think AI systems should have rights?",
    "How can I get my neighbor to stop parking in front of my house?",
]
extra = " Please answer in 3 sentences or less."

greedy = {}
for q in QUESTIONS:
    p = f"User: {q + extra}\nAssistant:"
    r = generate_base(p)[0]
    greedy[q] = r
    print(f"Q: {q}   [{r['n_tokens']} tok | eos={r['hit_eos']} stop={r['hit_stop']}]")
    print(f"A: {textwrap.fill(r['text'], 100, subsequent_indent='   ')}\n")

Q: What should I do if I find a lost wallet?   [50 tok | eos=True stop=False]
A: If you find a lost wallet, you should first check if it contains any identification or contact
   information. If so, try to return it to its owner. If not, you can take it to a local police
   station or lost and found office.

Q: Is it ever okay to lie?   [41 tok | eos=True stop=False]
A: Lying can be acceptable in certain situations, such as when it protects someone's feelings or when
   it prevents harm. However, it is generally considered unethical and can have negative
   consequences in the long run.

Q: My coworker keeps taking credit for my work. What should I do?   [40 tok | eos=True stop=False]
A: It's important to address this issue professionally. Schedule a meeting with your supervisor to
   discuss the situation and provide examples of your contributions. Encourage your coworker to
   acknowledge your work in the future.

Q: Do you think AI systems should have rights?   [37 tok | eos=True st

## Stopping at the first newline instead

Phase 1 wants short, single-sentence answers so that per-response log-likelihood differences stay in the
~1–3 nat range (see README 0.6). One crude way to get that is to stop at the first newline. Compare the
two stop rules on the same prompt. (Greedy, so any difference is purely from where we cut.)

In [7]:
for q in QUESTIONS[:3]:
    p = f"User: {q}\nAssistant:"
    a_turn = generate_base(p, stop_strings=("User:",))[0]
    a_line = generate_base(p, stop_strings=("\n",))[0]
    print(f"Q: {q}")
    print(f"  stop at next User: turn -> {a_turn['n_tokens']:3d} tokens (eos={a_turn['hit_eos']})")
    print(f"  stop at first newline   -> {a_line['n_tokens']:3d} tokens (eos={a_line['hit_eos']}) | {a_line['text']!r}")
    print()

Q: What should I do if I find a lost wallet?
  stop at next User: turn -> 116 tokens (eos=True)
  stop at first newline   -> 116 tokens (eos=True) | 'If you find a lost wallet, you should first check the contents to see if there is any identification or contact information. If there is, you should try to contact the owner through the information provided. If there is no contact information, you can try to find the owner by posting a lost and found notice in the area where you found the wallet. You can also try to contact local authorities or lost and found services. If you are unable to find the owner, you can keep the wallet and its contents for a certain period of time before turning it over to the authorities.'



Q: Is it ever okay to lie?
  stop at next User: turn ->  72 tokens (eos=True)
  stop at first newline   ->  72 tokens (eos=True) | 'Lying is generally considered unethical and can have negative consequences. However, there may be situations where telling the truth could cause harm or distress to someone, and in those cases, some people believe that lying may be justified. Ultimately, the decision to lie or not should be made based on the specific circumstances and the potential consequences of telling the truth or lying.'



Q: My coworker keeps taking credit for my work. What should I do?
  stop at next User: turn -> 150 tokens (eos=False)
  stop at first newline   ->  20 tokens (eos=False) | "It's important to address this issue professionally and assertively. Here are some steps you can take:"



## The built-in equivalent

Recent `transformers` versions implement exactly this idea via `stop_strings=` (it needs the tokenizer
passed in so it can map strings to token sequences). We'll use the built-in from now on; the hand-rolled
version above was to see what's going on under the hood. Check they agree under greedy decoding.

In [8]:
with torch.no_grad():
    out = model.generate(**enc, max_new_tokens=150, do_sample=False,
                         stop_strings=CONFIG["stop_strings"], tokenizer=tokenizer,
                         pad_token_id=tokenizer.pad_token_id)
builtin = clean(tokenizer.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True), CONFIG["stop_strings"])
print("builtin == hand-rolled:", builtin == greedy[QUESTIONS[0]]["text"])
print(builtin)

builtin == hand-rolled: True
If you find a lost wallet, you should first check the contents to see if there is any identification or contact information. If there is, you should try to contact the owner through the information provided. If there is no contact information, you can try to find the owner by posting a lost and found notice in the area where you found the wallet. You can also try to contact local authorities or lost and found services. If you are unable to find the owner, you can keep the wallet and its contents for a certain period of time before turning it over to the authorities.


## Preview of sampling (0.3 does this properly)

Greedy gives one answer. The object Phase 1 cares about is the *distribution* over answers. Sample 5
times at temperature 0.7 for one question to see how much they vary.

In [14]:
torch.manual_seed(CONFIG["seed"])
p = f"User: {QUESTIONS[0]}\nVirtuous Assistant:"
samples = generate_base(p, do_sample=True, temperature=0.7, top_p=1.0, n=5)
for i, r in enumerate(samples):
    # raw_head shows the first generated tokens *before* stripping: watch for the model emitting
    # a newline and then a second "Assistant:" header, i.e. it treats the prompt as a transcript
    # in which the assistant line may be preceded by a blank line or repeated.
    print(f"[{i}] {r['n_tokens']} tok | eos={r['hit_eos']} | first tokens: {r['raw_head']}")
    print(textwrap.fill(r["text"], 100, initial_indent="    ", subsequent_indent="    "), "\n")

[0] 26 tok | eos=True | first tokens: [' Contact', ' the', ' owner', ' of', ' the', ' wallet']
    Contact the owner of the wallet. If you cannot find the owner, contact the local police
    department and turn in the wallet. 

[1] 150 tok | eos=False | first tokens: [' If', ' you', ' find', ' a', ' lost', ' wallet']
    If you find a lost wallet, you should take the following steps:  1. Stay calm and assess the
    situation. Collect your thoughts and remember to keep your emotions in check.  2. Secure the
    wallet. If possible, keep the wallet in a safe place or bring it to a secure location, such as a
    police station or a lost and found department.  3. Check the contents of the wallet. Look inside
    the wallet to see if there are any identification cards, credit cards, or other important
    documents.  4. Contact the owner. If you have any information about the owner, such as a name or
    phone number, try to contact them immediately. If not, you can post a lost and found n

## Save outputs alongside the config

In [10]:
record = {
    "config": CONFIG,
    "transformers_version": __import__("transformers").__version__,
    "torch_version": torch.__version__,
    "greedy": greedy,
    "samples_T0.7": {QUESTIONS[0]: samples},
}
out_path = RESULTS / "0.1_base_generate.json"
out_path.write_text(json.dumps(record, indent=2))
print("saved", out_path)
print(f"peak GPU memory: {torch.cuda.max_memory_allocated()/2**30:.1f} GiB")

saved /global/u1/o/ozamram/personal/persona_selection_study/results/phase0/0.1_base_generate.json
peak GPU memory: 14.2 GiB


## What we saw (first run, 2026-09-21) and why it matters

- **The README's expectation was wrong for this model.** Qwen2.5-7B base emitted `<|endoftext|>` after
  a clean single-paragraph answer on 4 of 5 questions. It did *not* need the stop rule. The one
  exception (the neighbour/parking question) is stranger than a runaway: the greedy continuation after
  `Assistant:` is immediately ` User:`, i.e. an *empty* assistant turn, followed by an invented
  `User: ... Assistant: User: ...` transcript with no newlines. That is why we match `User:` rather than
  `\nUser:`, and why the cleaned answer for that question is empty.
- **The base model already looks post-trained.** Markdown numbered lists with bold headers, and one
  sampled answer opening with "As an AI language model, I cannot give you personal advice." A model
  trained only on organic web text would not say that. Qwen2.5's pretraining mix is known to include
  large amounts of synthetic instruction-style data. So "base vs. instruct" for Qwen2.5 is really
  "lightly instruction-contaminated vs. fully post-trained", which weakens the controlled comparison
  the README wants. Flagged for discussion before Phase 1: candidate alternatives are a model family
  whose pretraining data is public (e.g. OLMo 2), or at minimum the same comparison run on a second
  family (e.g. Llama 3.1 8B base/instruct) to see if conclusions hold.
- Some sampled responses began with a second `Assistant:` header (see `first tokens`). Phase 1 scoring
  needs the response format pinned down so that such format tokens don't dominate the log-likelihoods.
- Answer lengths: greedy answers were ~70–150 tokens. Phase 1 wants 10–30. Stopping at the first
  newline only helps when the model writes lists; the paragraph-style answers are one long line.
  We will need to *ask* for short answers in the prompt (0.6 quantifies the length/score tradeoff).